### Notebook 范围

### 目的
按 case 绘制 O3、U60N10、U60N50 evolution，并加入 BWCN reference 和 B2000 climatology。

### 科学问题
不同 hindcast case 中，臭氧路径和极涡风速路径是否与同年 reference 偏离？ensemble spread 是否与 reference/climatology 的位置有关？

### 方法
O3 使用 60-90N、30-70 hPa partial column；U 使用 60N zonal mean，10 hPa 和 50 hPa。thin lines 是成员，粗线和阴影是 ensemble mean ±1 std，黑线是 BWCN 同年 reference，灰虚线是 B2000 climatology。

### 输出
outputs/figures/02_evolution/<year>/ 和 outputs/tables/02_evolution/<year>/。

### 解读
如果 ensemble spread 先在 U 中扩大，再在 O3 中扩大，支持动力过程上游控制臭氧不确定性。

### 注意
不同初始化月份没有一月/二月完整数据；x 轴统一用月份坐标，但曲线从各 case 初始化月开始。


### 导入与公共路径

### 目的
加载 Extention_analysis 的公共函数、数据路径、输出目录和绘图参数。

### 科学问题
保证所有 notebook 使用同一套变量定义，避免 EPFlux、O3、U、Z300 的窗口和符号约定互相漂移。

### 方法
使用 hindcast_extension_utils.py 中的 DATA_ROOT、HINDCAST_ROOT、BWCN_ROOT、B2000_ROOT、FIG_DIR、TAB_DIR、CACHE_DIR 和 LOG_DIR。

### 输出
无图；只打印工作目录和 Hindcast 数据目录。

### 解读
如果路径打印正确，后续代码块会使用同一套 cleaned hindcast 产品。

### 注意
所有写入都限制在 code_cleaned/Hindcast_experiment/Extention_analysis/outputs 下；不会修改原始数据。


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 绘制多 case evolution

### 目的
为每个 case 生成一张三联图：O3、U60N10、U60N50。

### 科学问题
加入 reference 和 climatology 后，hindcast ensemble 是偏向 reference、climatology，还是形成独立路径？

### 方法
按年份分目录保存；每张图有同名 CSV，记录 ensemble mean/std、reference 和 climatology。

### 输出
02_evolution_<case>_O3_U60N10_U60N50_v2.png/pdf。

### 解读
黑线接近 ensemble mean 表示 skill 好；灰虚线用于判断异常是否只是 climatological seasonal cycle。

### 注意
BWCN reference U60 是从原始 U 插值到 pressure level 后 zonal mean；B2000 climatology 来自已生成的 climatology pressure-level 文件。


In [ ]:
def _member_mean_std(da):
    return da.mean("member", skipna=True), da.std("member", skipna=True)

def _plot_member_evolution(ax, da, date, case, color, label):
    x = case_time_doy(case, date[: da.sizes["lead_time"]])
    for i in range(da.sizes["member"]):
        ax.plot(x, da.isel(member=i).values, color=color, alpha=0.16, lw=0.6)
    mean, std = _member_mean_std(da)
    ax.plot(x, mean.values, color=color, lw=2.4, label=label)
    ax.fill_between(x, (mean - std).values, (mean + std).values, color=color, alpha=0.20, linewidth=0)
    return x, mean, std

def _plot_reference(ax, ref, date, case, label="BWCN reference"):
    if ref is None:
        return None
    x = case_time_doy(case, date[: ref.sizes["lead_time"]])
    ax.plot(x, ref.values, color="black", lw=2.1, label=label)
    return x

def _plot_clim(ax, clim, start_doy, end_doy, label="B2000 climatology"):
    if clim is None:
        return
    sub = clim.sel(doy=slice(start_doy, end_doy))
    ax.plot(sub["doy"].values, sub.values, color="0.35", ls="--", lw=1.8, label=label)

fig_overview_rows = []
inv = discover_hindcast_cases()
o3_clim = load_b2000_o3_partial_climatology()
u10_clim = load_b2000_u60_climatology(10)
u50_clim = load_b2000_u60_climatology(50)

for _, row in inv.iterrows():
    case = row["case"]; year = row["year"]
    fig_dir = figure_dir("02_evolution", year)
    tab_dir = table_dir("02_evolution", year)
    o3, o3_date = load_hindcast_o3(case)
    if o3 is None:
        log_message(f"{case}: skip evolution, missing O3")
        continue
    ref_o3, ref_o3_date = load_bwcn_reference_o3(year)
    u10, u10_date = compute_u60(case, 10)
    u50, u50_date = compute_u60(case, 50)
    ref_u10, ref_u10_date = load_bwcn_reference_u60(year, 10)
    ref_u50, ref_u50_date = load_bwcn_reference_u60(year, 50)
    start_doy = init_doy_for_case(case)
    end_doy = mmdd_to_doy(5, 30)
    fig, axes = plt.subplots(3, 1, figsize=(10.5, 8.8), sharex=True, constrained_layout=True)
    table_rows = []

    x, mean, std = _plot_member_evolution(axes[0], o3, o3_date, case, "tab:blue", "Hindcast ens mean ±1σ")
    _plot_reference(axes[0], ref_o3, ref_o3_date, f"{year}-01")
    _plot_clim(axes[0], o3_clim, start_doy, end_doy)
    axes[0].set_ylabel("O3 (DU)\n60-90N, 30-70 hPa")
    axes[0].set_title(f"{case}: O3 and vortex wind evolution")
    for d, m, s in zip(x, mean.values, std.values):
        table_rows.append({"case": case, "variable": "O3_60_90N_30_70hPa", "doy": int(d), "ensemble_mean": float(m), "ensemble_std": float(s)})

    for ax, da, date, ref, ref_date, clim, plev, color in [
        (axes[1], u10, u10_date, ref_u10, ref_u10_date, u10_clim, 10, "tab:orange"),
        (axes[2], u50, u50_date, ref_u50, ref_u50_date, u50_clim, 50, "tab:green"),
    ]:
        if da is None:
            ax.text(0.5, 0.5, f"U60N{plev} missing", transform=ax.transAxes, ha="center", va="center")
            continue
        x, mean, std = _plot_member_evolution(ax, da, date, case, color, "Hindcast ens mean ±1σ")
        _plot_reference(ax, ref, ref_date, f"{year}-01")
        _plot_clim(ax, clim, start_doy, end_doy)
        ax.axhline(0, color="0.25", lw=0.8)
        if plev == 50:
            ax.axhline(U60_THRESH, color="tab:red", lw=1.0, ls=":", label="FWD threshold 7 m/s")
        ax.set_ylabel(f"U60N{plev} (m/s)")
        for d, m, s in zip(x, mean.values, std.values):
            table_rows.append({"case": case, "variable": f"U60N{plev}", "doy": int(d), "ensemble_mean": float(m), "ensemble_std": float(s)})

    for ax in axes:
        ax.axvline(start_doy, color="0.2", lw=1.0, ls="-.", alpha=0.8)
        ax.legend(loc="best", fontsize=8, ncol=2)
        ax.grid(True, alpha=0.25)
    set_month_axis(axes[-1], start_doy, end_doy)
    table = pd.DataFrame(table_rows)
    csv = tab_dir / f"02_evolution_{case}_O3_U60N10_U60N50.csv"
    table.to_csv(csv, index=False)
    fig_overview_rows.append({"case": case, "year": year, "n_points": len(table), "figure_dir": str(fig_dir)})
    savefig(fig, f"02_evolution_{case}_O3_U60N10_U60N50_v2", fig_dir=fig_dir, notebook="02_multicase_O3_U_evolution.ipynb", scientific_question="O3 和 vortex wind pathway 是否同步偏离 reference/climatology？", variables_windows="O3: 60-90N 30-70hPa; U: 60N 10/50hPa; init-May30; black=BWCN same-year; gray dashed=B2000 climatology", interpretation="U spread 早于 O3 spread 支持 dynamics-first；reference/climatology 用于判断偏差方向。", caveat="later-initialized cases 不包含初始化前月份，不能解释 long-lead precursor。", csv_table=csv)
    plt.close(fig)

summary = pd.DataFrame(fig_overview_rows)
summary_csv = table_dir("02_evolution") / "02_multicase_evolution_summary.csv"
summary.to_csv(summary_csv, index=False)
summary.to_csv(TAB_DIR / "02_multicase_evolution_summary.csv", index=False)
print(summary.to_string(index=False))
write_figure_guide()
